# NRTK Test Stage With GPU Models
[RI Issue 288](https://gitlab.jatic.net/jatic/reference-implementation/reference-implementation/-/issues/288) sets the objective of using GPU acceleration wherever possible within the NRTK test stage.  In short, there is nothing NRTK-specific to do here (see Background) but this notebook does demonstrate how an NRTK test stage performance will improve when the model under test is on GPU.

## Background

NRTK consists of:

* **Image Perturbers** which, when called, return a perturbed image (and modified bounding boxes when applicable, as most perturbations will relocate boxes)
* **Perturber Factories** which yield a series of Image Perturbers based on one or more theta keys (e.g. a series of perturbers with an increasing blur factor)
* **Model Evaluation** which includes some metrics, scorers, and generators for assessing model performance with regards to perturbations

The Reference Implementation (RI) does not use any of the NRTK Model Evaluation components as they are (generally speaking) alternative implementations of MAITE-native workflows.  The `PerturbImage` implementations fall into two categories - (1) Kitware's PyBSM package for physics-based simulations of real sensor perturbations, and (2) generic perturbations which are wrappers of tools in `scikit-image`, `pillow`, and `opencv2`.  None of these can be executed on GPU.

## Set Up

This notebook will:

* Instantiate an RI-wrapped Visdrone model provided by Kitware twice- with the only distinction that one is loaded onto CPU and the other GPU.
* Wrap a single image into an RI Dataset
* Instantiate an NRTK Test Stage which includes a pertuber factory - one theta key with five levels (meaning model inference will run five times on perturbed iterations of the singleton dataset)
* Execute the Test Stage on both GPU and CPU

## Key Points

The two main takeaways are:

* We compare performance of CPU and GPU, noting that at least with Mac MPS this workflow is about 5x faster
* We show a "reasonable" NRTK result, namely that our metric (mAP) decreases as the perturber theta key (kernel size of an average blur) increases

Lastly, although not a key part of this demo... there is a "third takeaway".  This notebook also includes an option in the code below to toggle the perturber factory to a PyBSM-based perturber.  The parameters for the PyBSM sensor and scenario are highly technical and finding a configuration that executes (let alone produces a resonable result) is difficult (many of the config values tried simply caused the Python kernel to crash).  However, the configuration embedded below _does_ at least execute and we can see that each perturbation requires 5-10 seconds to complete on a single image.  If we can validate that a 'resonable' PyBSM configuration results in 10+ seconds per image perturbation, we should raise the issue with Kitware about possible new features for GPU acceleration or other optimization.

### Instantiate the models
First, as a basis for comparison, we create two of the same model, except one of them will be loaded onto our GPU device.

In [ ]:
from jatic_ri.object_detection.models import VisdroneODModel

# The notebook author only tested 'mps' but presumably this would work for 'cuda' as well.
GPU_DEVICE = 'mps'

# This will copy a ~110 MB .pth file to your CWD
model_cpu = VisdroneODModel(
    arch = "resnet50",
    model_pickle_dir = None, # TODO - this param should default to None in the RI wrapper
    device = "cpu"
)
 
model_gpu = VisdroneODModel(
    arch = "resnet50",
    model_pickle_dir = None,
    device = GPU_DEVICE
)   

### Create the dataset and metric
Load a dataset of one image.  This notebook is partially derived from [this demo](https://github.com/Kitware/nrtk/blob/main/docs/examples/coco_scorer.ipynb) where Kitware has provided a single Visdrone image and an annotation file in COCO format.  Two changes had to be made manually to the annotation file:

1. Added "img_gsd" to the image metadata (required for a PyBSM perturber)
2. Added a class id 0.  Required until issue #324 is resolved

We also instantiate the basic mAP metric.

Visualize the image with its bounding boxes overlaid to validate.

In [ ]:
import json
import os
import urllib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from jatic_ri.object_detection.datasets import CocoDetectionDataset
from jatic_ri.object_detection.metrics import map50_torch_metric_factory

# Creates ./data directory in your PWD and downloads one JPG and one JSON file into it
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

img_url = "https://data.kitware.com/api/v1/item/623880f14acac99f429fe3ca/download"
img_path = os.path.join(DATA_DIR, "visdrone_img.jpg")
if not os.path.isfile(img_path):
    _ = urllib.request.urlretrieve(img_url, img_path)  # noqa: S310

# The original source for this annotation file is "https://data.kitware.com/api/v1/item/6596fde99c30d6f4e17c9eff/download".
# Two changes had to be made manually:
#    (1) Added "img_gsd" to the image metadata (required for a PyBSM perturber)
#    (2) Added a class id 0.  Required until issue #324 is resolved

dataset = CocoDetectionDataset(root = DATA_DIR, ann_file = os.path.join("sample.json"))

metric = map50_torch_metric_factory()

# Display the one image with GT boxes
fig, ax = plt.subplots()
ax.set_title("GT Detections")
ax.imshow(np.asarray(dataset[0][0]).transpose(1,2,0))
ax.set_axis_off()
for bbox in dataset[0][1].boxes:
    width = bbox[2] - bbox[0]
    height = bbox[3] - bbox[1]

    ax.add_patch(
        Rectangle(
            (bbox[0], bbox[1]),
            width,
            height,
            linewidth=1,
            edgecolor="b",
            facecolor="none",
        ),
    )
plt.show()

### Prediction Using MAITE Evalute (No NRTK Test Stage... just for demonstration)
This is a simple example using the `evaluate` workflow to generate predictions across the entire "dataset" of one image to validate that the model, dataset, and metric are wrapped correctly and that the GPU is being used.

The completion time will be printed for CPU and GPU, and the predictions will be displayed overlaid on the image.

Your mileage may vary, but the performance difference on the author's Mac (M3 Pro) with MPS was:

```
1it [00:01,  1.30s/it]
On CPU, time to evaluate was 1.3173727989196777
1it [00:00,  4.05it/s]
On GPU, time to evaluate was 0.2611961364746094
```


In [ ]:
# Evaluate one image as a test and show the predictions
from maite.tasks import evaluate
import time

start = time.time()
_, _, _ = evaluate(
    model=model_cpu,
    dataset=dataset,
    metric=metric,
    batch_size=1,
    return_augmented_data=False,
    return_preds=True,
)
print(f"On CPU, time to evaluate was {time.time() - start}")

start = time.time()
mean_ap, pred, _ = evaluate(
    model=model_gpu,
    dataset=dataset,
    metric=metric,
    batch_size=1,
    return_augmented_data=False,
    return_preds=True,
)
print(f"On GPU, time to evaluate was {time.time() - start}")

fig, ax = plt.subplots()
ax.set_title(f"Model Prediction Detections, mAP: {mean_ap}")
ax.imshow(np.asarray(dataset[0][0]).transpose(1,2,0))
ax.set_axis_off()
for bbox in pred[0][0].boxes:
    width = bbox[2] - bbox[0]
    height = bbox[3] - bbox[1]

    ax.add_patch(
        Rectangle(
            (bbox[0], bbox[1]),
            width,
            height,
            linewidth=1,
            edgecolor="r",
            facecolor="none",
        ),
    )
plt.show()

### Wrapping and Testing in NRTK

As mentioned in the introduction, the NRTK tools that the RI uses (image perturbers) are not written to execute on GPU.  So, the only performance gains that will be seen in an `NRTKTestStage` are due to faster model inference when the MAITE-wrapped model is loaded onto GPU.

So we will create an NRTK test stage with a factory that will yield five different perturbers.  This will lead to the test stage running the `evaluate` on the (singleton) dataset five times.  Given that the single `evaluate` call ran roughly 1 second faster on GPU, we expect the whole test stage to run about 5 seconds faster.

Indeed, the author on Mac observed similar performance:
```
1it [00:01,  1.27s/it]
1it [00:01,  1.25s/it]
1it [00:01,  1.21s/it]
1it [00:01,  1.21s/it]
1it [00:01,  1.21s/it]
On CPU, time to complete test stage was 6.213612079620361
1it [00:00,  3.81it/s]
1it [00:00,  4.63it/s]
1it [00:00,  4.63it/s]
1it [00:00,  4.63it/s]
1it [00:00,  4.62it/s]
On GPU, time to complete test stage was 1.1903302669525146
```

### Side Observation - PyBSM Performance

Also note below that we have embedded both generic and PyBSM-backed perturber factory configurations.  The generic is used by default since PyBSM takes additional time to execute which obscures the increased performance of running inference on GPU.  (We also don't have a set of theta values which can bAlso, neither perburber (i.e. a type of augmentation) can currently be run on GPU.  As mentioned in the introduction, if the PyBSM perturber takes ~10 seconds per image perturbation in a real use case, we should raise the concern with Kitware.

In [ ]:
from jatic_ri.object_detection.test_stages import NRTKTestStage

# TODO: Nathan Doll noticed perturbations using PyBSM on 256x256 images exceeded 30s.  I need to swap the generic below for PyBSM and see if it affects numbers...

nrtk_config_generic = {
    "name": "NRTKTestStage Example",
    "perturber_factory": {
        "type": "nrtk.impls.perturb_image_factory.generic.step.StepPerturbImageFactory",
        "nrtk.impls.perturb_image_factory.generic.step.StepPerturbImageFactory": {
            "perturber": "nrtk.impls.perturb_image.generic.cv2.blur.AverageBlurPerturber",
            "theta_key": "ksize",
            "start": 1,
            "stop": 10,
            "step": 2,
        },
    },
}

nrtk_config_pybsm = {
    "name": "natural_robustness_",
    "perturber_factory": {
        "type": "nrtk.impls.perturb_image_factory.pybsm.CustomPybsmPerturbImageFactory",
        "nrtk.impls.perturb_image_factory.pybsm.CustomPybsmPerturbImageFactory": {
            "sensor": {
                "type": "nrtk.impls.perturb_image.pybsm.sensor.PybsmSensor",
                "nrtk.impls.perturb_image.pybsm.sensor.PybsmSensor": {
                    "name": "L32511x",
                    "D": 275e-3,
                    "f": 4,
                    "p_x": 0.008e-3,
                    "opt_trans_wavelengths": [
                        0.5e-6,
                        0.66e-6
                    ],
                    "optics_transmission": [0.5,0.5],
                    "eta": 0.4,
                    "int_time": 0.03,
                    "dark_current": 0,
                    "read_noise": 25.0,
                    "max_n": 96000,
                    "bit_depth": 11.9,
                    "max_well_fill": 0.6,
                    "s_x": 0.0005e-3,
                    "s_y": 0.0005e-3,
                    "da_x": 100e-6,
                    "da_y": 100e-6,
                    "qe": [
                        0.05,
                        0.6,
                        0.75,
                        0.85,
                        0.85,
                        0.75,
                        0.5,
                        0.2,
                        0
                    ],
                    "qe_wavelengths": [
                        3e-07,
                        4e-07,
                        5e-07,
                        6e-07,
                        7e-07,
                        8e-07,
                        9e-07,
                        1e-06,
                        1.1e-06
                    ]
                }
            },
            "scenario": {
                "type": "nrtk.impls.perturb_image.pybsm.scenario.PybsmScenario",
                "nrtk.impls.perturb_image.pybsm.scenario.PybsmScenario": {
                    "name": "niceday",
                    "ihaze": 1,
                    "altitude": 9000,
                    "ground_range": 0,
                    "aircraft_speed": 100
                }
            },
            "thetas": [
                [
                    1,
                    2,
                    3
                ]
            ],
            "theta_keys": [
                "f"
            ]
        }        
    }
}

# Change to use PyBSM perturber above
CONFIG = nrtk_config_generic

test_stage_cpu = NRTKTestStage()
# Don't emulate this hardcoding of <component>_id in real uses... just a workaround
test_stage_cpu.load_model(model_id="model_id", model=model_cpu)
test_stage_cpu.load_dataset(dataset_id="dataset_id", dataset=dataset)
test_stage_cpu.load_metric(metric_id="metric_id", metric=metric)
start = time.time()
test_stage_cpu.run(use_stage_cache=False, config = CONFIG, threshold=0.5)
print(f"On CPU, time to complete test stage was {time.time() - start}")

test_stage_gpu = NRTKTestStage(),
# Don't emulate this hardcoding of <component>_id in real uses... just a workaround
test_stage_gpu.load_model(model_id="model_id", model=model_gpu)
test_stage_gpu.load_dataset(dataset_id="dataset_id", dataset=dataset)
test_stage_gpu.load_metric(metric_id=metric.return_key, metric=metric)
start = time.time()
test_stage_gpu.run(use_stage_cache=False, config = CONFIG, threshold=0.5)
print(f"On GPU, time to complete test stage was {time.time() - start}")

### Validating NRTK Test Stage Reasonableness

Lastly, the outputs of one of the test stages are printed below.

The configuration assigned to `ARGS` above defines a perturber factory that genertates pertubers that apply an average blur on the elements of the dataset with kernel sizes ranging from 1 pixel (equivalent to no blur) to 9 pixels.

It is reasonable for the mAP score of `perturbation_0` (where `ksize=1`, =~ no blur) to equal the mAP score on the standalone `evaluate` call in *Prediction Using MAITE Evalute* above.  As the kernel size (and, therefore, "bluriness") increases with perturbations 1-4, mAP score drops accordingly.

### Side Observation - Perturbation Names

Note that the names of the perturbations in the output dictionary simply become `perturbation_n`, losing any information about the specific theta keys and values.  This is particularly likely to confuse in the case where multiple theta keys are included in the factory.  For example, if variable A is parameterized to [1,2,3] and B to ["white","grey","black"], then there will be nine perturbers yielded - (1, "white"), (1, "grey"), (1, "black), (2, "white)", etc. - but the two dimensions will be collapsed to `perturber_[0-8]` in the output.

In [ ]:
test_stage_gpu.outputs